**Test1 with small dataset**

In [1]:
# Essential Librraries
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import drive
import os

In [2]:
# Step 1: Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Unzip the file
!unzip -q '/content/drive/MyDrive/Colab Notebooks/Agri-tech/Dataset-test.zip' -d /content/dataset/

In [4]:
# Define paths (Update these to match actual Drive folder names)
train_dir = '/content/dataset/test-dataset/train'
valid_dir = '/content/dataset/test-dataset/valid'

# Image properties
IMG_SIZE = (224, 224) # MobileNetV2 standard input size
BATCH_SIZE = 32

In [5]:
# --- STEP 3: FAST DATA LOADING (tf.data) ---
# This is significantly faster than ImageDataGenerator
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

# Prefetching allows the CPU to prepare the next batch while the GPU trains the current one
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 4032 files belonging to 1 classes.
Found 1008 files belonging to 1 classes.


In [6]:
# --- STEP 4: MODEL BUILDING (MOBILE NET V2) ---
# This was the missing section causing your "NameError"
num_classes = len(os.listdir(train_dir)) # Automatically count your plant categories

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Add a preprocessing layer because MobileNetV2 expects pixels between -1 and 1
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(preprocess_input), # Built-in preprocessing
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
# --- STEP 5: TRAINING ---
# Ensure you have set Runtime -> Change runtime type -> T4 GPU
history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 157ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/st

In [8]:
# --- STEP 6: EXPORT FOR MOBILE (TFLite) ---
model.save('plant_disease_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

# Generate the labels.txt file for your mobile app
class_names = sorted(os.listdir(train_dir))
with open('labels.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print("Success! Your .tflite model and labels.txt are ready for mobile.")

Saved artifact at '/tmp/tmp3n0ku7u9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138981136726352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981136726928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981136726544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981136727888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981136727696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981136726736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981116445712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981116446672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981116446288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138981116444752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1389811164

Test the Module

In [11]:
import numpy as np
import tensorflow as tf
from google.colab import files
from tensorflow.keras.preprocessing import image
import os

# 1. Load the model with the custom object fix
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
model = tf.keras.models.load_model(
    'plant_disease_model.h5',
    custom_objects={'preprocess_input': preprocess_input}
)

# 2. Get the Class Names
# If the dataset folder is missing, we define them manually or check the directory
data_path = '/content/dataset/New Plant Diseases Dataset(Augmented)/train'

if os.path.exists(data_path):
    class_names = sorted(os.listdir(data_path))
    print(f"Classes found: {class_names}")
else:
    # If the folder is missing, list the names manually based on your dataset
    # Example: class_names = ['Apple_Scab', 'Apple_Cedar_Rust', 'Healthy']
    print("Warning: Training folder not found. Please ensure you unzipped the data.")
    # Fallback: using folder names from your original project setup
    class_names = ['Apple_Cedar_Rust'] # Add your other class names here

# 3. Upload and Predict
uploaded = files.upload()

for filename in uploaded.keys():
    img = image.load_img(filename, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Match model input (1, 224, 224, 3)

    predictions = model.predict(img_array)

    # Argmax finds the index of the highest probability
    result_index = np.argmax(predictions[0])

    if result_index < len(class_names):
        predicted_class = class_names[result_index]
        confidence = np.max(predictions[0]) * 100
        print(f"\nResult: {predicted_class} ({confidence:.2f}%)")
    else:
        print("\nError: Prediction index out of range. Check your class_names list.")

Saving Pear-trellis-rust-on-Comice-pear-23aba02.jpg to Pear-trellis-rust-on-Comice-pear-23aba02.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

Result: Apple_Cedar_Rust (100.00%)
